<a href="https://colab.research.google.com/github/PhucPower300121/FLUX-Jupyter/blob/main/flux1_kontext_t2i_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### FLUX.1 Kontext [dev] Text-to-Image (t2i) — Colab (ComfyUI + GGUF Q4)

Credit: [ComfyUI](https://github.com/comfyanonymous/ComfyUI), [ComfyUI-GGUF (city96)](https://github.com/city96/ComfyUI-GGUF), [FLUX.1-Kontext-dev-GGUF (unsloth)](https://huggingface.co/unsloth/FLUX.1-Kontext-dev-GGUF)

In [ ]:
#@title Install ComfyUI + ComfyUI-GGUF + download model
%cd /content/
!git clone https://github.com/comfyanonymous/ComfyUI

%cd /content/ComfyUI

PIN_COMMIT = ""  # để trống = bản mới nhất, hoặc điền hash commit cụ thể để cố định version
if PIN_COMMIT:
    !git fetch --all -q
    !git reset --hard {PIN_COMMIT}

!pip install -q -r requirements.txt

# Cài custom node GGUF
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/city96/ComfyUI-GGUF comfyui_gguf
!pip install -q -r comfyui_gguf/requirements.txt

%cd /content/ComfyUI
import os
os.makedirs("/content/ComfyUI/models/unet", exist_ok=True)
os.makedirs("/content/ComfyUI/models/vae", exist_ok=True)
os.makedirs("/content/ComfyUI/models/text_encoders", exist_ok=True)

!apt -y install -qq aria2

# GGUF Q4_K_S, mmap load -> không đè RAM system trước khi qua GPU
unet_path = '/content/ComfyUI/models/unet/flux1-kontext-dev-Q4_K_S.gguf'
vae_path = '/content/ComfyUI/models/vae/ae.safetensors'
clip_l_path = '/content/ComfyUI/models/text_encoders/clip_l.safetensors'
t5xxl_path = '/content/ComfyUI/models/text_encoders/t5xxl_fp8_e4m3fn_scaled.safetensors'

if not os.path.exists(unet_path):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/unsloth/FLUX.1-Kontext-dev-GGUF/resolve/main/flux1-kontext-dev-Q4_K_S.gguf -d /content/ComfyUI/models/unet -o flux1-kontext-dev-Q4_K_S.gguf
if not os.path.exists(vae_path):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Lumina_Image_2.0_Repackaged/resolve/main/split_files/vae/ae.safetensors -d /content/ComfyUI/models/vae -o ae.safetensors
if not os.path.exists(clip_l_path):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors -d /content/ComfyUI/models/text_encoders -o clip_l.safetensors
if not os.path.exists(t5xxl_path):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o t5xxl_fp8_e4m3fn_scaled.safetensors

from IPython.display import clear_output
clear_output()

paths = [unet_path, vae_path, clip_l_path, t5xxl_path]
missing = [p for p in paths if not os.path.exists(p)]
if missing:
    print("\033[91mMISSING FILE (Rerun this cell):\033[0m", missing)
else:
    print("\033[92mInstall + download model complete.\033[0m")

In [ ]:
#@title Load model to RAM (GGUF, mmap -- low RAM)
%cd /content/ComfyUI

import random, torch, numpy as np
from PIL import Image
from nodes import NODE_CLASS_MAPPINGS
import nodes

# Nạp toàn bộ node built-in (comfy_extras) + custom node (GGUF...) qua cơ chế chính thức của ComfyUI
from nodes import init_extra_nodes
await init_extra_nodes()

UnetLoaderGGUF = NODE_CLASS_MAPPINGS["UnetLoaderGGUF"]()
DualCLIPLoader = NODE_CLASS_MAPPINGS["DualCLIPLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
ConditioningZeroOut = NODE_CLASS_MAPPINGS["ConditioningZeroOut"]()
ReferenceLatent = NODE_CLASS_MAPPINGS["ReferenceLatent"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
VAEEncode = NODE_CLASS_MAPPINGS["VAEEncode"]()
LoadImage = nodes.LoadImage()

with torch.inference_mode():
    clip = DualCLIPLoader.load_clip("clip_l.safetensors", "t5xxl_fp8_e4m3fn_scaled.safetensors", "flux")[0]
    unet = UnetLoaderGGUF.load_unet("flux1-kontext-dev-Q4_K_S.gguf")[0]
    vae = VAELoader.load_vae("ae.safetensors")[0]

print("Model load complete.")

In [ ]:
#@title Run Text-to-Image
positive_prompt = "a photo of a cat sitting on a windowsill"  #@param {type:"string"}
negative_prompt = ""  #@param {type:"string"}
width = 1024  #@param {type:"slider", min:256, max:1536, step:32}
height = 1024  #@param {type:"slider", min:256, max:1536, step:32}
steps = 20  #@param {type:"slider", min:4, max:40, step:1}
cfg = 1.0  #@param {type:"number"}
sampler_name = "euler"  #@param ["euler", "euler_ancestral", "dpmpp_2m", "dpmpp_2m_sde"]
scheduler = "simple"  #@param ["simple", "normal", "karras"]
seed = 0  #@param {type:"integer"}

# Kontext la model distilled, cfg mac dinh 1.0 (negative prompt vo tac dung o cfg=1.0).
# De negative prompt hoat dong, tang cfg len 2.0-4.0.

with torch.inference_mode():
    if seed == 0:
        seed = random.randint(0, 18446744073709551615)
    print("Seed:", seed)

    positive = CLIPTextEncode.encode(clip, positive_prompt)[0]
    if negative_prompt.strip():
        negative = CLIPTextEncode.encode(clip, negative_prompt)[0]
    else:
        negative = ConditioningZeroOut.zero_out(positive)[0]

    lat_h, lat_w = height // 8, width // 8
    empty_latent = {"samples": torch.zeros([1, 16, lat_h, lat_w])}

    samples = KSampler.sample(
        unet, seed, steps, cfg, sampler_name, scheduler,
        positive, negative, empty_latent, denoise=1.0
    )[0]

    decoded = VAEDecode.decode(vae, samples)[0].detach()
    out_img = Image.fromarray(np.array(decoded * 255, dtype=np.uint8)[0])
    out_img.save("/content/output_t2i.png")

out_img

In [ ]:
#@title Download output (t2i)
from google.colab import files
files.download("/content/output_t2i.png")